In [6]:
import numpy as np
from pydrake.all import (
    StartMeshcat, 
    MeshcatVisualizer, 
    DiagramBuilder, 
    AddMultibodyPlantSceneGraph, 
    Parser,
    Simulator
)

def simulate_swinging_pendulum():
    # 启动 Meshcat 可视化服务器
    meshcat = StartMeshcat()
    
    # 构建包含物理引擎和几何场景的系统图 (Diagram)
    builder = DiagramBuilder()
    plant, scene_graph = AddMultibodyPlantSceneGraph(builder, time_step=0.0) 

    # 解析并加载单摆模型，随后锁定物理参数
    parser = Parser(plant)
    parser.AddModels(url="package://drake/examples/pendulum/Pendulum.urdf")
    plant.Finalize()
    
    # 连接可视化器并生成系统图
    MeshcatVisualizer.AddToBuilder(builder, scene_graph, meshcat)
    diagram = builder.Build()
    
    # 实例化仿真器
    simulator = Simulator(diagram)
    
    # 获取仿真器管理的上下文 (Context) 以修改初始状态
    context = simulator.get_mutable_context()
    plant_context = plant.GetMyMutableContextFromRoot(context)
    
    # 设置初始状态：水平位置 (pi/2 rad)，静止释放
    plant.SetPositions(plant_context, [np.pi / 2.0])
    plant.SetVelocities(plant_context, [0.0])
    
    # 设置仿真速度与挂钟时间同步，避免计算过快导致无法观测
    simulator.set_target_realtime_rate(1.0) 
    
    print(f"Meshcat URL: {meshcat.web_url()}")
    print("Starting simulation...")
    
    # 执行仿真
    simulator.AdvanceTo(60.0) 
    
    print("Simulation complete.")

if __name__ == "__main__":
    simulate_swinging_pendulum()

INFO:drake:Meshcat listening for connections at http://localhost:7004


Meshcat URL: http://localhost:7004
Starting simulation...
Simulation complete.
